In [0]:
bronze_df = spark.read.table("workspace.default.bronze_overdose")

display(bronze_df)
print(f"Total rows: {bronze_df.count()}")

State,Indicator,Date,Death_Count,Percent_Complete,Percent_Pending_Investigation,State_Name,Predicted_Value
AK,Cocaine (T40.5),2016-04-01,11,100,0.0,Alaska,11
AK,Cocaine (T40.5),2016-05-01,11,100,0.0,Alaska,11
AK,Cocaine (T40.5),2016-06-01,11,100,0.0,Alaska,11
AK,Cocaine (T40.5),2016-07-01,13,100,0.0,Alaska,13
AK,Cocaine (T40.5),2016-08-01,12,100,0.02338634238,Alaska,12
AK,Cocaine (T40.5),2016-09-01,11,100,0.0469924812,Alaska,11
AK,Cocaine (T40.5),2016-10-01,13,100,0.06991377301,Alaska,13
AK,Cocaine (T40.5),2016-11-01,15,100,0.06994637445,Alaska,15
AK,Cocaine (T40.5),2016-12-01,15,100,0.06888633754,Alaska,15
AK,Cocaine (T40.5),2017-01-01,15,100,0.06858710562,Alaska,15


Total rows: 47365


In [0]:
# clean the data
# 1. Death_Count and Predicted_Value to numeric (they may have come in as strings)
# 2. Date to proper date type
# 3. drop nulls in key columns
# 4. drop duplicates

from pyspark.sql.functions import col, to_date, regexp_replace
from pyspark.sql.types import DoubleType, IntegerType

silver_df = bronze_df \
    .withColumn("Death_Count", 
                regexp_replace(col("Death_Count"), ",", "").cast(DoubleType())) \
    .withColumn("Predicted_Value", 
                regexp_replace(col("Predicted_Value"), ",", "").cast(DoubleType())) \
    .withColumn("Date", to_date(col("Date"), "yyyy-MM-dd")) \
    .withColumn("Percent_Complete", col("Percent_Complete").cast(DoubleType())) \
    .withColumn("Percent_Pending_Investigation", 
                col("Percent_Pending_Investigation").cast(DoubleType())) \
    .dropna(subset=["State", "Indicator", "Date", "Death_Count"]) \
    .dropDuplicates()

print(f"Bronze rows: {bronze_df.count()}")
print(f"Silver rows: {silver_df.count()}")
display(silver_df)

Bronze rows: 47365
Silver rows: 47365


State,Indicator,Date,Death_Count,Percent_Complete,Percent_Pending_Investigation,State_Name,Predicted_Value
AK,Cocaine (T40.5),2016-04-01,11.0,100.0,0.0,Alaska,11.0
AK,Cocaine (T40.5),2016-05-01,11.0,100.0,0.0,Alaska,11.0
AK,Cocaine (T40.5),2016-06-01,11.0,100.0,0.0,Alaska,11.0
AK,Cocaine (T40.5),2016-07-01,13.0,100.0,0.0,Alaska,13.0
AK,Cocaine (T40.5),2016-08-01,12.0,100.0,0.02338634238,Alaska,12.0
AK,Cocaine (T40.5),2016-09-01,11.0,100.0,0.0469924812,Alaska,11.0
AK,Cocaine (T40.5),2016-10-01,13.0,100.0,0.06991377301,Alaska,13.0
AK,Cocaine (T40.5),2016-11-01,15.0,100.0,0.06994637445,Alaska,15.0
AK,Cocaine (T40.5),2016-12-01,15.0,100.0,0.06888633754,Alaska,15.0
AK,Cocaine (T40.5),2017-01-01,15.0,100.0,0.06858710562,Alaska,15.0


In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_overdose")

print("Silver table created successfully")
print(f"Columns: {silver_df.columns}")

Silver table created successfully
Columns: ['State', 'Indicator', 'Date', 'Death_Count', 'Percent_Complete', 'Percent_Pending_Investigation', 'State_Name', 'Predicted_Value']
